# 04 · Feed-forward networks for local runoff

How might the same rainfall produce different runoff in two catchments? We’ll build a small **artificial neural network (ANN)** to learn a synthetic rainfall–runoff relationship. We'll follow each layer by hand, reuse Lesson 3’s training functions, and connect the resulting model to the encoders and output layer in DROP.

This lesson uses made-up data with a known response. Our target is **local runoff depth in mm/day**. In DROP, runoff generated locally is later routed through the river network to form a discharge prediction. We’ll add time dependence and routing in later lessons.

## 1 · See what one hidden unit does

A **neuron** or **unit** takes a weighted sum of the inputs, adds a bias, then applies an **activation function**. A **dense** layer gives each unit its own weight for every input. A **hidden layer** is an intermediate layer. We train it through the final prediction error, so its individual values don’t need measured targets.

Imagine a unit with weight 1 on scaled rainfall and bias −0.5. Its number before activation is rainfall minus 0.5. **ReLU**, short for rectified linear unit, returns zero for negative inputs and leaves positive inputs alone. With rainfall scaled by 10, an input of 0.3 means 3 mm/day. The unit produces −0.2 before activation and zero after. At 8 mm/day it produces 0.3 after activation.

**Try it.** Predict the three numbers after ReLU. Which rainfall values give an output of zero?

In [ ]:
import torch  # Provides tensors and the ReLU operation.

# Three synthetic rainfall depths divided by 10. One value per row here.
rain = torch.tensor([0.3, 0.5, 0.8])  # (3,)
before_relu = rain - 0.5  # Subtract the same scalar from each entry. (3,) → (3,).
after_relu = torch.relu(before_relu)  # Clamp negatives to zero. Shape stays (3,).
print("Before activation", before_relu)  # Shows each weighted sum plus bias.
print("After activation ", after_relu)   # Shows the unit's three outputs.

<details>
<summary>Show solution</summary>

After ReLU the values are 0, 0, and 0.3. Rainfall at or below 5 mm/day gives zero in this hand-built unit.

</details>

## 2 · Why put an activation between layers?

A feed-forward network sends inputs through successive layers. A **multilayer perceptron (MLP)** is a feed-forward network built from dense layers with nonlinear activations. Without an activation, chaining two dense layers still gives one weighted sum plus bias per output.

Take a first layer that gives `2x + 1` and a second that gives `3h − 2`. Substituting the first expression for `h` produces `6x + 1`. If we place ReLU between the layers, the result becomes `3 × max(0, 2x + 1) − 2`. Below `x = −0.5` it is flat. Above that value it rises. The network can now represent changes in slope, which a single straight line cannot capture.

**Try it.** At `x = −1`, calculate both outputs. Then open the model code and find where ReLU enters the forward path.

<details>
<summary>Show solution</summary>

The two dense layers give −5. With ReLU the first layer’s −1 becomes zero and the second layer returns −2. In `FeatureEncoder`, ReLU follows `nn.Linear` inside `nn.Sequential`.

</details>

## 3 · Trace a batch through the model

Each row below is one catchment-day. Features are scaled daily rainfall and a synthetic wetness-retention index that stays fixed for each catchment. Higher index values raise the rainfall threshold in this toy example. The **encoder** turns the two inputs into a learned representation of four hidden values. The **prediction head** combines those four values into one number.

For a batch of two rows the shape path is `(2, 2) → (2, 4) → (2, 1) → (2,)`. Only the last axis changes inside the encoder. If we keep batch and time axes, the same encoder can map `(batch, time, 2)` to `(batch, time, 4)`. Applying it to a sequence still processes each time step separately. Learning across time will require the sequence model introduced later.

**Try it.** Predict the encoder’s output shape if you pass one catchment with three time steps and two features. Check by running the cell.

In [ ]:
import torch  # Provides sample tensors and controls gradient recording.
from pytorch_lab.nn.runoff_mlp import RunoffMLP  # Imports our model class.

# Initialise an encoder from 2 features to 4 hidden units and a head from 4 to 1.
model = RunoffMLP(num_features=2, hidden_size=4)
# Each row contains scaled rainfall and the index for one catchment-day.
rows = torch.tensor([[0.3, 0.2], [0.8, 0.6]])  # (2, 2)
series = torch.tensor([[[0.3, 0.2], [0.8, 0.2], [1.1, 0.2]]])  # (1, 3, 2)

# Inspect the outputs without recording gradients. This does not fit the model.
with torch.no_grad():
    encoded_rows = model.encoder(rows)  # (2, 2) → (2, 4)
    predictions = model(rows)           # (2, 2) → (2,), untrained runoff
    encoded_series = model.encoder(series)  # (1, 3, 2) → (1, 3, 4)

# Convert the shape objects to tuples to display just the axis lengths.
print(tuple(encoded_rows.shape), tuple(predictions.shape))
print(tuple(encoded_series.shape))

<details>
<summary>Show solution</summary>

The sequence encoder returns `(1, 3, 4)`. The first two axes still label catchment and day. Each pair of input features has become four hidden values.

</details>

## 4 · Train on two catchments and check later days

Our simulated observation is zero until rainfall reaches a catchment-specific threshold. Above the threshold, 0.8 mm of runoff is generated for every additional millimetre of rain. For index 0.2 the threshold is `2 + 5 × 0.2 = 3` mm/day. At 8 mm/day the target is `0.8 × (8 − 3) = 4` mm/day. For index 0.6 the threshold is 5 mm/day, so the same storm gives 2.4 mm/day. This simple rule leaves out antecedent moisture, snow, evapotranspiration, and travel time.

We generate 12 days for two catchments and hold out the last four days from both. Rainfall is divided by 10 using a fixed conversion. If you instead scale inputs by statistics such as their mean and standard deviation, estimate those statistics from training rows and reuse them for validation. This keeps future readings out of the fitting procedure.

**Try it.** Calculate both catchments’ targets for the first validation day, when rain is 5 mm/day. Then run the fit and compare both predictions with your numbers.

In [ ]:
import torch  # Provides tensors, least squares, and the optimizer.
from pytorch_lab.nn.runoff_mlp import RunoffMLP
from pytorch_lab.training import training_step, validation_step

# Use the same initial parameters each time this cell runs.
torch.manual_seed(1)

rain = torch.tensor([1., 3., 5., 8., 11., 2., 6., 10., 5., 7., 12., 4.])
index = torch.tensor([0.2, 0.6])  # Fixed synthetic attribute for each catchment.
# Make two (12, 2) tables. Every row is a day and every column is a catchment.
rain_grid = rain[:, None].expand(-1, 2)   # Repeat each day's rain across catchments.
index_grid = index[None, :].expand(12, -1)  # Repeat both attributes across days.
# Stack along a new final axis so each catchment-day has two features.
features = torch.stack((rain_grid / 10, index_grid), dim=-1)  # (12, 2, 2)
# Generate the known answer per catchment-day by broadcasting and clipping.
targets = 0.8 * (rain_grid - (2 + 5 * index_grid)).clamp_min(0)  # (12, 2)

# Split by day before flattening days and catchments into sample rows.
x_train, y_train = features[:8].reshape(-1, 2), targets[:8].reshape(-1)  # (16, 2), (16,)
x_val, y_val = features[8:].reshape(-1, 2), targets[8:].reshape(-1)       # (8, 2), (8,)

# Fit a straight-line baseline using the two features plus a column for the bias.
train_design = torch.cat((x_train, torch.ones_like(x_train[:, :1])), dim=1)  # (16, 3)
val_design = torch.cat((x_val, torch.ones_like(x_val[:, :1])), dim=1)  # (8, 3)
coefficients = torch.linalg.lstsq(train_design, y_train).solution  # Three fitted numbers.
linear_val = val_design @ coefficients  # Weighted sum of each row, shape (8,).

# Create a new MLP and use full-batch SGD with the shared training step.
model = RunoffMLP(num_features=2, hidden_size=4)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
for _ in range(1500):
    training_step(model, x_train, y_train, optimizer)  # One parameter update per call.

# Score both models on the same eight observed targets.
mlp_val, mlp_mse = validation_step(model, x_val, y_val)  # Vector (8,) and scalar ().
linear_mse = (linear_val - y_val).square().mean()  # Eight errors reduced to one score.
print(f"Linear MSE {linear_mse.item():.5f}, MLP MSE {mlp_mse.item():.5f} (mm/day)²")
# The first two validation rows represent the same day in the two catchments.
print("First-day observations", y_val[:2])
print("First-day MLP predictions", mlp_val[:2])

<details>
<summary>Show solution</summary>

The first catchment gives 1.6 mm/day and the second gives zero. An MLP prediction can differ from either target. Compare its errors on both catchments and the overall MSE. This controlled example tests whether the model can learn a threshold-like response.

</details>

**Try it.** Set `torch.manual_seed(7)` in the training cell and rerun it. Which model's score changes? What does this tell you about comparing models using a single run? Reset the seed to 1 afterwards.

<details>
<summary>Show solution</summary>

The MLP starts with different weights and can learn a different response. In this example seed 7 gives a higher validation MSE than the linear baseline. The least-squares baseline fits the same line to the same training rows. Report the settings used, and compare several initialisations before drawing a general conclusion.

</details>

## 5 · Find these pieces in DROP

Open `../drop-ch/project/nn/layers/lstm.py` and look for `EncodingModule` and `LSTM`.

| Our module | Corresponding role in DROP |
| --- | --- |
| `FeatureEncoder` applies a stack to the last axis | `EncodingModule` builds a configurable stack of linear, activation, and dropout layers |
| Our feature row combines rainfall and a catchment attribute | DROP has separate dynamic and static encoders. The dynamic encoder preserves the time axis. The static encoding gets a length-one time axis and broadcasts across the sequence when added to the dynamic encoding. |
| `RunoffMLP.head` converts hidden features to one output per row | `LSTM.out_layer` converts recurrent hidden features to output values at each time step. |
| `training_step` calls the model and updates its weights | `RunoffNet` connects batch data, runoff predictions, and loss through Lightning hooks. |

**Try it.** If three catchments each have eight days and the encoders emit four features, what shapes come out of the dynamic and static encoders? What is the shape just after adding the encodings?

<details>
<summary>Show solution</summary>

The dynamic encoder returns `(3, 8, 4)`. The static encoder returns `(3, 4)`, which becomes `(3, 1, 4)` after inserting a time axis. Adding them broadcasts the static features across eight days and gives `(3, 8, 4)` for the recurrent layer.

</details>

### Optional lecture reading

In [HODL 01](HODL_01_Introduction.ipynb), revisit *Constructing Neural Networks* through *Learning a Boolean Function*. Pay particular attention to linear layers, activations, assembling a model, and what the training loop updates. The Boolean example gives a second setting in which the choice of activation matters. Read the dropout subsection alongside DROP's encoder code. Save softmax and image classification for when you need classification.

In [HODL 02](HODL_02_Computer_Vision_and_Audio.ipynb), *Revisiting Dense Neural Networks* is optional reinforcement if you want another view of layers and overfitting. Return to the RNN and LSTM sections of [HODL 03](HODL_03_Natural_Language_Processing.ipynb) in Lessons 5–6. Use *Graph Representation* in [HODL 04](HODL_04_Graph_Neural_Networks.ipynb) before the routing lesson. The graph notation transfers directly, while DROP's routing and water balance come from its own code. **HODL 05** covers a separate topic and can wait.